# 미니 장미 생장 단계 판단 — 혼합형 (Colab)

2단계(등급 판단)를 **YOLO 봉오리 위치 검출 → FastSAM 봉오리 마스크 → 분류 모델 등급** 으로 나눈 버전입니다. 자세한 설명: `HYBRID.md`

**처음 한 번만**
1. `rose_hybrid` 폴더를 구글 드라이브 **내 드라이브** 바로 아래에 통째로 업로드합니다.
2. 드라이브에서 `rose_hybrid.ipynb` 를 더블클릭 → **Google Colaboratory** 로 엽니다.
3. 학습(5번)을 하려면 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택합니다.

그다음 위에서부터 셀을 차례로 실행합니다 (셀 왼쪽 ▶ 또는 `Shift+Enter`).

| 단계 | 내용 | 필요할 때 |
|---|---|---|
| 0 | 준비 (드라이브 연결) | 매번 |
| 1 | 입력 사진 확인·교체 | 매번 |
| 2 | 구멍 위치 보정 | 처음 1회, 카메라를 옮겼을 때 |
| 3 | 구멍 위치·빈 구멍 점수 확인 | 보정 후 |
| 4 | 1단계만 실행 (검출 모델 없이) | 확인용 |
| 5 | 혼합형 모델 학습 (위치 검출 + 등급 분류) | 처음 1회, 데이터가 바뀌었을 때 |
| 6 | 전체 실행 | 매번 |
| 7 | 결과 내려받기 | 필요할 때 |

## 0. 준비

In [ ]:
# 이 폴더를 업로드한 위치 (다른 곳에 올렸다면 수정)
PROJECT_DIR = '/content/drive/MyDrive/rose_hybrid'

import os, sys, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:          # 로컬 Jupyter 에서는 노트북이 있는 폴더를 사용
    IN_COLAB = False
    PROJECT_DIR = os.path.abspath('.')

os.chdir(PROJECT_DIR)
for p in (PROJECT_DIR, os.path.join(PROJECT_DIR, 'tools')):
    if p not in sys.path:
        sys.path.insert(0, p)

from IPython.display import Image, display

def latest_run():
    runs = sorted(glob.glob('output/2*'))
    return runs[-1] if runs else None

def show(paths, width=900):
    for p in paths:
        print(p)
        display(Image(p, width=width))

print('작업 폴더:', os.getcwd())
print(sorted(os.listdir()))

In [ ]:
# 봉오리 검출·FastSAM·분류(YOLOv11)용 라이브러리 설치 — 런타임을 새로 시작할 때마다 필요
%pip install -q ultralytics

In [ ]:
# (선택) 동작 확인: 합성 이미지로 전체 흐름 테스트 — '통과' 가 나오면 정상
!python tests/test_pipeline.py
!python tests/test_hybrid.py

## 1. 입력 사진

`input/CL.jpg`, `input/CR.jpg` 를 사용합니다. 새 사진으로 바꾸려면 `UPLOAD = True` 로 바꾸고 실행한 뒤
**CL.jpg, CR.jpg** 이름의 파일을 선택하세요.

In [ ]:
UPLOAD = False

if UPLOAD and IN_COLAB:
    from google.colab import files
    for name, data in files.upload().items():
        with open(os.path.join('input', name), 'wb') as f:
            f.write(data)
        print('저장:', name)

show(['input/CL.jpg', 'input/CR.jpg'], width=600)

## 2. 구멍 위치 보정 (처음 1회)

사진 위에서 **R1C1 → R1C2 → … → R3C4** 순서로 구멍 중심을 클릭하고 **[저장]** 을 누릅니다.
- 빈 구멍도 보이면 찍습니다 (식물 유무를 자동 판단할 위치).
- 잎에 가려진 구멍은 화분 위치를 짐작해 찍습니다.
- 사진 가장자리 밖 구멍은 **회색 여백**에 짐작해 찍습니다 (그 식물 잎이 이웃 식물로 섞이는 것을 막음).
- 아예 알 수 없는 구멍은 **[건너뛰기]**.
- 이미 저장된 점이 있으면 처음부터 표시됩니다. 점을 드래그해서 고칠 수 있습니다.

In [ ]:
from calibrate_colab import calibrate
calibrate('CL')

In [ ]:
calibrate('CR')

## 3. 구멍 위치·빈 구멍 점수 확인

빨간 원 = 식물, 파란 원 = 빈 구멍. 점수가 `presence.empty_ratio`(기본 0.38) 이상이면 빈 구멍입니다.
판단이 틀리면 `configs/pipeline.yaml` 의 `presence` 값을 조정하세요.

In [ ]:
!python tools/preview_holes.py
show(['output/holes_preview_CL.jpg', 'output/holes_preview_CR.jpg'])

## 4. 1단계만 실행 (검출 모델 없이)

식물 유무와 식물별 분할만 확인합니다. 모든 식물이 `grade_zero` 로 나옵니다.

In [ ]:
!python run_pipeline.py --no-detector
show(sorted(glob.glob(f'{latest_run()}/vis_*.jpg')))

## 5. 혼합형 모델 학습 (GPU 권장)

혼합형은 모델 3개를 씁니다. **①과 ③만 학습**하고 ②는 공개 가중치를 그대로 씁니다(첫 실행 때 자동으로 내려받음).

| | 모델 | 하는 일 | 학습 결과 |
|---|---|---|---|
| ① | YOLO 검출 (1클래스) | 봉오리 **위치**만 찾음 (등급 구분 없음) | `runs/detect/rose_bud1/weights/best.pt` |
| ② | FastSAM | 박스 안 봉오리의 정확한 **마스크** | (학습 안 함) `FastSAM-s.pt` |
| ③ | YOLO 분류 | 봉오리 크롭의 **등급** (`not_bud` / `grade_one`~`grade_four`) | `runs/classify/rose_grade/weights/best.pt` |

`dataset.zip` 을 Colab 로컬 디스크에 풀고, ①용(라벨을 모두 '봉오리' 한 클래스로)·③용(박스를 잘라 등급 폴더로) 데이터로 바꾼 뒤 학습합니다.
학습 결과는 드라이브에 저장되어, 다음에는 이 단계를 건너뛰어도 됩니다.

In [ ]:
# 5-1. 학습 데이터 준비 (①용 det/, ③용 cls/)
import zipfile, torch

DATA_ROOT = '/content/dataset'          # dataset.zip 을 풀 곳
HYBRID_DATA = '/content/hybrid_data'    # 혼합형 학습 데이터
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('학습 장치:', DEVICE)

if not os.path.exists(os.path.join(DATA_ROOT, 'data.yaml')):
    zipfile.ZipFile('dataset.zip').extractall(DATA_ROOT)

EXTRA = '--extra-cls extra_cls' if os.path.isdir('extra_cls') else ''   # 5-4 에서 직접 고른 크롭
!python tools/prepare_hybrid_data.py --data-root {DATA_ROOT} --out {HYBRID_DATA} {EXTRA}

In [ ]:
# 5-2. ① 봉오리 위치 검출 학습 (더 정확하게: yolo11s.pt, 느려짐)
!python train_hybrid.py det --data {HYBRID_DATA} --model yolo11n.pt --epochs 100 --device {DEVICE}

In [ ]:
# 5-3. ③ 등급 분류 학습
!python train_hybrid.py cls --data {HYBRID_DATA} --model yolo11n-cls.pt --epochs 50 --device {DEVICE}

In [ ]:
# 학습 곡선(①)·분류 혼동 행렬(③)
show([p for p in ['runs/detect/rose_bud1/results.png',
                  'runs/classify/rose_grade/confusion_matrix_normalized.png'] if os.path.exists(p)])

### 5-4. (선택) 실제 트레이 사진으로 분류 모델 보강

`not_bud` 는 학습 이미지에서 봉오리가 없는 곳을 잘라 만든 것이라, 반사판·케이블·손 같은 **트레이 환경의 오탐지는 배우지 못했습니다.**
1. 아래 셀을 실행하면 `input/` 사진에서 ①이 찾은 후보를 잘라 `mined/<예측 등급>/` 에 저장합니다. 사진이 더 있으면 경로를 바꾸세요.
2. 드라이브에서 `mined/` 를 열고, 틀린 크롭을 맞는 폴더로 옮깁니다. 봉오리가 아니면 `not_bud` 로 옮깁니다.
3. 폴더 이름을 `extra_cls` 로 바꾸고 5-1, 5-3 을 다시 실행합니다.

In [ ]:
!python tools/mine_crops.py input/*.jpg --out mined --device {DEVICE}

## 6. 전체 실행

결과는 `output/<시각>/` 에 저장됩니다: `result_buds.jpg`, `result.json`, `table.txt`, `vis_CL.jpg`, `vis_CR.jpg`, `masks/`.
표에서 식물 = 등급, 빈 구멍 = `-`, 판단 불가 = `?`.

봉오리 표시: 박스 = 위치, **흰 윤곽선 = FastSAM 봉오리 마스크**, `G2 0.83 -> R1C1` = 분류 등급 · 분류 확신 · 배정된 칸.
`result.json` 의 봉오리마다 `det_conf`(① 위치 신뢰도)와 `mask_px`(마스크 픽셀 수)가 함께 저장됩니다.

**`result_buds.jpg` = 최종 결과판**: 칸마다 최종 등급과 그 칸에 배정된 봉오리 사진을 높은 등급부터 보여 줍니다. `table.txt` 에는 카메라별 봉오리 개수가 함께 저장됩니다.

In [ ]:
!python run_pipeline.py
show([f'{latest_run()}/result_buds.jpg'], width=1100)
show(sorted(glob.glob(f'{latest_run()}/vis_*.jpg')))

## 7. 결과 내려받기

In [ ]:
import shutil

run = latest_run()
if run:
    base = '/content' if IN_COLAB else 'output'
    archive = shutil.make_archive(os.path.join(base, os.path.basename(run)), 'zip', run)
    print('압축:', archive)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)